### 0 操作准备


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
import os
print(os.getcwd())

In [ ]:
# Use a single source dataframe (ms)
ms = pd.read_csv("../data/MT_UPDATE_MS_HEV.csv")
tf_ms = ms.copy()
tf_ms.head()


In [ ]:
print(ms.dtypes)

### 1 风险指数阈值探索


1.1 查看风险指标字段与基本统计


In [ ]:
tf_index_risk_equal = "TF_Index_Risk_equal"   # 目标风险指数字段

print("Column name:", tf_index_risk_equal)
print("Missing values:", ms[tf_index_risk_equal].isna().sum())
print("Min:", ms[tf_index_risk_equal].min())
print("Max:", ms[tf_index_risk_equal].max())
print("Mean:", ms[tf_index_risk_equal].mean())
print("Median:", ms[tf_index_risk_equal].median())

ms[tf_index_risk_equal].describe()

1.2 绘制风险指数分布


In [ ]:
plot_s = pd.to_numeric(ms[tf_index_risk_equal], errors="coerce").dropna()

plt.figure(figsize=(10, 5))
plt.hist(plot_s, bins=50, color="#4C72B0", edgecolor="white", alpha=0.9)

# 对 y 轴（Number of street segments）做对数化
plt.yscale("log")

plt.xlabel("Risk Index")
plt.ylabel("Number of street segments (log scale)")
plt.title("Distribution of Risk Index (log-scaled y-axis)")
plt.tight_layout()
plt.show()


1.3 不同阈值下高风险占比


In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]

summary = []
total_n = len(ms)

for t in thresholds:
    n = (ms[tf_index_risk_equal] > t).sum()
    pct = n / total_n * 100
    summary.append({
        "threshold": t,
        "count_above_threshold": n,
        "percentage_above_threshold": round(pct, 2)
    })

threshold_summary = pd.DataFrame(summary)
threshold_summary

In [ ]:
ms[tf_index_risk_equal].quantile([0.5, 0.7, 0.8, 0.85, 0.9, 0.95, 0.98, 0.99])

In [ ]:
thresholds = [0.20, 0.22, 0.24, 0.25, 0.26, 0.28, 0.30]

summary = []
total_n = len(ms)

for t in thresholds:
    n = (ms[tf_index_risk_equal] > t).sum()
    pct = n / total_n * 100
    summary.append({
        "threshold": t,
        "count_above_threshold": n,
        "percentage_above_threshold": round(pct, 2)
    })

threshold_summary = pd.DataFrame(summary)
threshold_summary

### 2 高风险样本筛选与特征整理


2.1 主表字段检查


In [ ]:
# Single-source workflow: use ms only
tf_ms = ms.copy()

print("MS_ID in ms:", "MS_ID" in ms.columns)
print("fid in ms:", "fid" in ms.columns)
print("MS_ID unique in ms:", ms["MS_ID"].nunique())
print("tf_ms shape:", tf_ms.shape)


In [ ]:
# Validate required columns directly in ms (no extra merge)
new_cols = [
    "N_TFlossR_2kiw",
    "N_TFlossR_2ktw",
    "N_TFlossR_2kmw"
]

tf_ms = ms.copy()
missing = [c for c in new_cols if c not in tf_ms.columns]
if missing:
    raise KeyError(f"Missing required columns in ms: {missing}")

print("Original ms shape:", ms.shape)
print("tf_ms shape:", tf_ms.shape)
print(tf_ms[new_cols].isna().sum())

tf_ms[["MS_ID"] + new_cols].head()
tf_ms.head()


2.2 以 risk index = 0.25 筛选高风险路段


In [ ]:

threshold = 0.25
label_threshold = 0.28    # 标签构建使用的更严阈值
risk_col = "TF_Index_Risk_equal"

high_risk_tf_ms = tf_ms[tf_ms[risk_col] > threshold].copy()

print("Training threshold:", threshold)
print("Number of high-risk MS streets:", len(high_risk_tf_ms))
print("Percentage of all MS streets:", round(len(high_risk_tf_ms) / len(ms) * 100, 2), "%")

high_risk_tf_ms.columns.tolist()
high_risk_tf_ms.dtypes.sort_index()
high_risk_tf_ms.head()

In [ ]:
meta_cols = [col for col in ["fid", "MS_ID", risk_col] if col in high_risk_tf_ms.columns] # join back to QGIS
meta_tf_ms = high_risk_tf_ms[meta_cols].copy()

meta_tf_ms.head()

In [ ]:
for col in high_risk_tf_ms.columns:
    print(col)

2.3 提取核心指标特征


In [ ]:
tf_simple_cols = [
    "TF_Index_population",
    "TF_Index_acc_loss",
    "TF_Index_centrality",
    "Tem_extent",
]

tf_simple_cols = [col for col in tf_simple_cols if col in high_risk_tf_ms.columns]

print("Simple feature columns:")
print(tf_simple_cols)

X_simple_tf_ms = high_risk_tf_ms[tf_simple_cols].copy()

print("X_simple shape:", X_simple_tf_ms.shape)
X_simple_tf_ms.head()

In [ ]:
print("Missing values in X_simple:")
print(X_simple_tf_ms.isna().sum())

constant_cols_simple_tf_ms = [col for col in X_simple_tf_ms.columns if X_simple_tf_ms[col].nunique() <= 1]
print("Constant columns in X_simple:", constant_cols_simple_tf_ms)

In [ ]:
X_simple_tf_ms.to_csv("../data/X_simple_tf_ms_0_25.csv", index=False)
meta_tf_ms.to_csv("../data/meta_tf_ms_0_25.csv", index=False)

2.4 提取人口结构细分指标


In [ ]:
population_detail_cols = ["TFResident_NOR",
                          "TFDaynight_NOR", 
                          "TFPRM_NOR"
]

acc_loss_detail_cols_tf = [
"N_TFlossR_2kiw", 
"N_TFlossR_2ktw", 
"N_TFlossR_2kmw"
]

structure_cols = [
    "TF_Index_centrality",
    "Tem_extent",
]

structure_cols = [col for col in structure_cols if col in high_risk_tf_ms.columns]
population_detail_cols = [col for col in population_detail_cols if col in high_risk_tf_ms.columns]
acc_loss_detail_cols_tf = [col for col in acc_loss_detail_cols_tf if col in high_risk_tf_ms.columns]

detailed_feature_cols_tf = population_detail_cols + acc_loss_detail_cols_tf + structure_cols

print("Detailed feature columns:")
print(detailed_feature_cols_tf)

X_detailed_tf_ms = high_risk_tf_ms[detailed_feature_cols_tf].copy()

print("X_detailed shape:", X_detailed_tf_ms.shape)
X_detailed_tf_ms.head()

In [ ]:
print("Missing values in X_detailed:")
print(X_detailed_tf_ms.isna().sum().sort_values(ascending=False))

constant_cols_detailed_tf_ms = [col for col in X_detailed_tf_ms.columns if X_detailed_tf_ms[col].nunique() <= 1]
print("Constant columns in X_detailed:", constant_cols_detailed_tf_ms)

In [ ]:
# 阈值：0.28

label_tf_ms = tf_ms[tf_ms[risk_col] > label_threshold].copy()


### 3 干预类型规则标注与建模


3.1 规则标签构建


3.1.1 三档分层规则


In [ ]:
col = "TF_Index_centrality"

print("col in df:", col in high_risk_tf_ms.columns)
print("similar cols:", [c for c in high_risk_tf_ms.columns if "centrality" in c.lower()])
print("dtype:", high_risk_tf_ms[col].dtype if col in high_risk_tf_ms.columns else None)

if col in high_risk_tf_ms.columns:
    s = pd.to_numeric(high_risk_tf_ms[col], errors="coerce")
    print("na:", s.isna().sum(), "nunique:", s.nunique(dropna=True))
    print("q33,q66:", s.quantile(0.33), s.quantile(0.66))


In [ ]:
# 将 4 个核心指标分成 low / medium / high

label_features = [
    "TF_Index_population",
    "TF_Index_centrality",
    "TF_Index_acc_loss",
    "Tem_extent",
]

def make_level_series(s, force_binary=False):
    if force_binary or set(s.dropna().unique()).issubset({0, 1}):
        return np.where(s > 0, "high", "low")
    
    q1 = s.quantile(0.33)
    q2 = s.quantile(0.66)
    
    return pd.cut(
        s,
        bins=[-np.inf, q1, q2, np.inf],
        labels=["low", "medium", "high"],
        include_lowest=True
    )

high_risk_tf_ms["pop_level"] = make_level_series(high_risk_tf_ms["TF_Index_population"])
high_risk_tf_ms["street_level"] = make_level_series(high_risk_tf_ms["TF_Index_centrality"])
high_risk_tf_ms["acc_level"] = make_level_series(high_risk_tf_ms["TF_Index_acc_loss"])
high_risk_tf_ms["exp_level"] = make_level_series(high_risk_tf_ms["Tem_extent"], force_binary=True)

In [ ]:
# 根据分层规则生成原因标签

def assign_cause_label(row):
    # Type B: Flood-exposed streets
    if (row["exp_level"] == "high") and (row["acc_level"] in ["medium", "high"]):
        return "B_flood_exposed"
    
    # Type A: One-and-only streets
    if (
        row["street_level"] == "high"
        and row["pop_level"] in ["medium", "high"]
        and row["acc_level"] in ["medium", "high"]
        and row["exp_level"] in ["low", "medium"]
    ):
        return "A_one_and_only"
    
    # Type C: No-service streets
    if (
        row["acc_level"] == "high"
        and row["street_level"] in ["low", "medium"]
        and row["exp_level"] == "low"
    ):
        return "C_no_service"
    
    return "mixed_review"

high_risk_tf_ms["cause_label"] = high_risk_tf_ms.apply(assign_cause_label, axis=1)

In [ ]:
# 生成干预标签（intervention label）

cause_to_intervention = {
    "A_one_and_only": 1,   # add street connection
    "B_flood_exposed": 2,  # elevated/protected street
    "C_no_service": 3,     # add emergency service point
    "mixed_review": -1
}

cause_to_intervention_name = {
    "A_one_and_only": "add_street_connection",
    "B_flood_exposed": "elevated_or_protected_street",
    "C_no_service": "add_emergency_service_point",
    "mixed_review": "review"
}

high_risk_tf_ms["intervention_label"] = high_risk_tf_ms["cause_label"].map(cause_to_intervention)
high_risk_tf_ms["intervention_name"] = high_risk_tf_ms["cause_label"].map(cause_to_intervention_name)

In [ ]:
# 先看每类样本量
print(high_risk_tf_ms["cause_label"].value_counts(dropna=False))
print()
print((high_risk_tf_ms["cause_label"].value_counts(normalize=True, dropna=False) * 100).round(2))

In [ ]:
# 去掉 mixed_review 样本，仅保留明确标签用于训练

train_tf_ms = high_risk_tf_ms[high_risk_tf_ms["cause_label"] != "mixed_review"].copy()

print("Training set size:", len(train_tf_ms))
print(train_tf_ms["cause_label"].value_counts())

3.1.2 规则更新版


In [ ]:
df = high_risk_tf_ms.copy()


pop_med = df["TF_Index_population"].quantile(0.33)
pop_high = df["TF_Index_population"].quantile(0.66)

acc_med = df["TF_Index_acc_loss"].quantile(0.33)
acc_high = df["TF_Index_acc_loss"].quantile(0.66)

# TF_Index_centrality 
street_medhigh = df["TF_Index_centrality"].quantile(0.33)
street_high = df["TF_Index_centrality"].quantile(0.66)

# Per_extent
df["is_exposed"] = df["Tem_extent"] > 0

print("pop_med, pop_high:", pop_med, pop_high)
print("acc_med, acc_high:", acc_med, acc_high)

# core label_v2

def assign_core_label_v2(row):
    # B: flood-exposed streets
    if row["is_exposed"] and row["TF_Index_acc_loss"] >= acc_med:
        return "B_flood_exposed"
    
    # A: one-and-only streets
    if (
        row["TF_Index_centrality"] >= street_high
        and row["TF_Index_population"] >= pop_med
        and row["TF_Index_acc_loss"] >= acc_med
        and not row["is_exposed"]
    ):
        return "A_one_and_only"
    
    # C: no-service streets
    if (
        row["TF_Index_acc_loss"] >= acc_high
        and row["TF_Index_centrality"] < street_high
        and not row["is_exposed"]
    ):
        return "C_no_service"
    
    return "mixed_review"

df["cause_label_core_v2"] = df.apply(assign_core_label_v2, axis=1)

print(df["cause_label_core_v2"].value_counts())
print((df["cause_label_core_v2"].value_counts(normalize=True) * 100).round(2))


3.1.3 连续得分规则（Type C）


In [ ]:
def minmax_norm(s):
    s_min = s.min()
    s_max = s.max()
    if s_max == s_min:
        return pd.Series(0.0, index=s.index)
    return (s - s_min) / (s_max - s_min)

df = label_tf_ms.copy()

df["street_n"] = minmax_norm(df["TF_Index_centrality"])
df["acc_n"] = minmax_norm(df["TF_Index_acc_loss"])
df["pop_n"] = minmax_norm(df["TF_Index_population"])
df["exp_n"] = df["Tem_extent"].astype(int)

df["score_A"] = 0.70 * df["street_n"] + 0.20 * df["pop_n"] + 0.10 * df["acc_n"]
df["score_C"] = 0.70 * df["acc_n"] + 0.20 * df["pop_n"] + 0.10 * (1 - df["street_n"])

def assign_cause_label_v3(row, dominance_margin=0.10, min_strength=0.45):
    # 1. 暴露路段优先判为 B
    if row["exp_n"] == 1:
        return "B_flood_exposed"
    
    # 2. 在高风险池中细分 A/C，其余样本记为 mixed_review
    diff = row["score_A"] - row["score_C"]
    
    if row["score_A"] >= min_strength and diff >= dominance_margin:
        return "A_one_and_only"
    
    if row["score_C"] >= min_strength and diff <= -dominance_margin:
        return "C_no_service"
    
    return "mixed_review"

df["cause_label_v3"] = df.apply(assign_cause_label_v3, axis=1)

print(df["cause_label_v3"].value_counts())
print((df["cause_label_v3"].value_counts(normalize=True) * 100).round(2))

In [ ]:
dominance_margin = 0.10
min_strength = 0.45

df["cause_label_v3b"] = df.apply(
    lambda row: assign_cause_label_v3(row, dominance_margin=0.08, min_strength=0.40),
    axis=1
)

print(df["cause_label_v3b"].value_counts())
print((df["cause_label_v3b"].value_counts(normalize=True) * 100).round(2))

In [ ]:
dm, ms = 0.08, 0.40
x = df[df["exp_n"] != 1].copy()
x["diff"] = x["score_A"] - x["score_C"]

cond_A = (x["score_A"] >= ms) & (x["diff"] >= dm)
cond_C = (x["score_C"] >= ms) & (x["diff"] <= -dm)
mixed = ~(cond_A | cond_C)

print("non-B total:", len(x))
print("A:", cond_A.sum(), "C:", cond_C.sum(), "mixed:", mixed.sum())

# mixed成因拆分
reason1 = mixed & (x["score_A"] < ms) & (x["score_C"] < ms)                 # 双低强度
reason2 = mixed & (x["score_A"] >= ms) & (x["score_C"] >= ms) & (x["diff"].abs() < dm)  # 双高但差距小
reason3 = mixed & (x["score_A"] >= ms) & (x["diff"] < dm)                    # A强但优势不足
reason4 = mixed & (x["score_C"] >= ms) & (x["diff"] > -dm)                   # C强但优势不足

print("both_weak:", reason1.sum())
print("both_strong_but_close:", reason2.sum())
print("A_strong_not_dominant:", reason3.sum())
print("C_strong_not_dominant:", reason4.sum())


In [ ]:
# ===== v4：直接在 risk > 0.2 的 high_risk_tf_ms 上按 v3b 规则打标 =====

# 你可以按需要微调这两个参数
dominance_margin = 0.08
min_strength = 0.40

df_v4 = high_risk_tf_ms.copy()

def minmax_norm(s):
    s = pd.to_numeric(s, errors="coerce")
    s_min, s_max = s.min(), s.max()
    if pd.isna(s_min) or pd.isna(s_max) or s_max == s_min:
        return pd.Series(0.0, index=s.index)
    return (s - s_min) / (s_max - s_min)

# 连续分数（不分档）
df_v4["street_n"] = minmax_norm(df_v4["TF_Index_centrality"])
df_v4["acc_n"] = minmax_norm(df_v4["TF_Index_acc_loss"])
df_v4["pop_n"] = minmax_norm(df_v4["TF_Index_population"])
df_v4["exp_n"] = (pd.to_numeric(df_v4["Tem_extent"], errors="coerce").fillna(0) > 0).astype(int)

df_v4["score_A"] = 0.70 * df_v4["street_n"] + 0.20 * df_v4["pop_n"] + 0.10 * df_v4["acc_n"]
df_v4["score_C"] = 0.70 * df_v4["acc_n"] + 0.20 * df_v4["pop_n"] + 0.10 * (1 - df_v4["street_n"])

def assign_cause_label_v4(row, dominance_margin=0.08, min_strength=0.40):
    # B 优先
    if row["exp_n"] == 1:
        return "B_flood_exposed"

    diff = row["score_A"] - row["score_C"]

    if row["score_A"] >= min_strength and diff >= dominance_margin:
        return "A_one_and_only"

    if row["score_C"] >= min_strength and diff <= -dominance_margin:
        return "C_no_service"

    return "mixed_review"

df_v4["cause_label_v4"] = df_v4.apply(
    lambda r: assign_cause_label_v4(r, dominance_margin=dominance_margin, min_strength=min_strength),
    axis=1
)

print("v4 label distribution (risk > 0.2):")
print(df_v4["cause_label_v4"].value_counts())
print((df_v4["cause_label_v4"].value_counts(normalize=True) * 100).round(2))

# 训练集：去掉 mixed（这里不会出现 merge 造成的 NaN）
train_tf_ms_v4 = df_v4[df_v4["cause_label_v4"] != "mixed_review"].copy()

print("\ntrain_tf_ms_v4 label distribution:")
print(train_tf_ms_v4["cause_label_v4"].value_counts())
print((train_tf_ms_v4["cause_label_v4"].value_counts(normalize=True) * 100).round(2))
print("\ntrain_tf_ms_v4 shape:", train_tf_ms_v4.shape)


In [ ]:
# 1. 从 df 中提取标签列
label_cols = ["MS_ID", "cause_label_v4"]
label_df = df_v4[label_cols].copy()
label_df.head()

# 2. 将标签 merge 回高风险主数据集
high_risk_labeled_tf_ms = high_risk_tf_ms.merge(label_df, on="MS_ID", how="left")

print(high_risk_labeled_tf_ms.shape)
print(high_risk_labeled_tf_ms["cause_label_v4"].value_counts(dropna=False))

# 3. 构建训练集：去除 mixed_review，仅保留明确标签
train_tf_ms = high_risk_labeled_tf_ms[
    high_risk_labeled_tf_ms["cause_label_v4"].notna() &
    (high_risk_labeled_tf_ms["cause_label_v4"] != "mixed_review")
].copy()

print(train_tf_ms["cause_label_v4"].value_counts())
print((train_tf_ms["cause_label_v4"].value_counts(normalize=True) * 100).round(2))


In [ ]:
# ===== 合并 v3b + v4 标签到同一个 high_risk_labeled_tf_ms 并导出 =====

# 1) v3b 标签来源：df（你前面生成 cause_label_v3b 的那个 df）
label_df_v3b = df[["MS_ID", "cause_label_v3b"]].drop_duplicates(subset=["MS_ID"]).copy()

# 2) v4 标签来源：df_v4（你前面生成 cause_label_v4 的那个 df_v4）
label_df_v4 = df_v4[["MS_ID", "cause_label_v4"]].drop_duplicates(subset=["MS_ID"]).copy()

# 3) 以 high_risk_tf_ms 为主表，左连接两套标签
high_risk_labeled_tf_ms = high_risk_tf_ms.copy()
high_risk_labeled_tf_ms = high_risk_labeled_tf_ms.merge(label_df_v3b, on="MS_ID", how="left")
high_risk_labeled_tf_ms = high_risk_labeled_tf_ms.merge(label_df_v4, on="MS_ID", how="left")

# 可选：看一下两列标签覆盖情况
print(high_risk_labeled_tf_ms[["cause_label_v3b", "cause_label_v4"]].isna().sum())
print(high_risk_labeled_tf_ms["cause_label_v3b"].value_counts(dropna=False))
print(high_risk_labeled_tf_ms["cause_label_v4"].value_counts(dropna=False))


In [ ]:
train_tf_ms.head()
high_risk_labeled_tf_ms.to_csv("../data/toGIS_tf_ms.csv", index=False)
train_tf_ms.to_csv("../data/train_tf_ms_0_25.csv", index=False)


In [ ]:
print(train_tf_ms.dtypes)

In [ ]:
# 4）用简化版特征先跑 baseline

simple_feature_cols = [
    "TF_Index_population",
    "TF_Index_acc_loss",
    "TF_Index_centrality",
    "Tem_extent",
]

X_train_simple_tf_ms = train_tf_ms[simple_feature_cols].copy()
y_train_tf_ms = train_tf_ms["cause_label_v4"].copy()

print(X_train_simple_tf_ms.shape)
print(y_train_tf_ms.value_counts())


In [ ]:
# 5. 训练集/测试集划分

from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_train_simple_tf_ms,
    y_train_tf_ms,
    test_size=0.2,
    random_state=42,
    stratify=y_train_tf_ms
)

print("Train shape:", X_tr.shape)
print("Test shape:", X_te.shape)
print("Train label distribution:")
print(y_tr.value_counts())
print("Test label distribution:")
print(y_te.value_counts())

In [ ]:
# 6. 模型训练与评估（baseline）

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced_subsample"
)

rf_model.fit(X_tr, y_tr)

In [ ]:
# 7）评估结果

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = rf_model.predict(X_te)

print("Accuracy:", accuracy_score(y_te, y_pred))
print()
print("Classification report:")
print(classification_report(y_te, y_pred))
print()
print("Confusion matrix:")
print(confusion_matrix(y_te, y_pred))

In [ ]:
# 8）看特征重要性

feature_importance = pd.DataFrame({
    "feature": X_train_simple_tf_ms.columns,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.bar(feature_importance["feature"], feature_importance["importance"])
plt.xticks(rotation=45)
plt.ylabel("Importance")
plt.title("Feature Importance - Random Forest Baseline")
plt.tight_layout()
plt.show()

1. Use PF_Index_Risk_equal > 0.25 to identify the high-risk street pool.
2. Use a stricter subset to generate clearer rule-based causal labels.
3. Exclude ambiguous mixed_review samples in the first supervised learning baseline.
4. Train a random forest classifier on simplified HEV-related indicators.
5. Predict the dominant intervention type for the broader high-risk street set.

3.2 训练干预类型模型
